# Chapter 5: Formatting Output and Speaking for the Model (Assistant prefilling)

- [Lesson](#lesson)
- [Exercises](#exercises)
- [Example Playground](#example-playground)

## Setup

Run the following setup cell to load your API key and establish the `get_completion` helper function.

In [ ]:
# Import python's built-in regular expression library
import re
import ollama

# Retrieve the MODEL_NAME variable from the IPython store
%store -r MODEL_NAME

# New argument added for prefill text, with a default value of an empty string
def get_completion(user_prompt: str, system_prompt="", prefill=""):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    if prefill:
        messages.append({"role": "assistant", "content": prefill})
    response = ollama.chat(
        model=MODEL_NAME,
        messages=messages,
        options={"temperature": 0.0, "num_predict": 2000},
    )
    return response["message"]["content"]

MODEL_NAME

---

## Lesson

**Large language models can format their output in a wide variety of ways**. You just need to ask them to do so!

One of these ways is by using XML tags to separate out the response from any other superfluous text. You've already learned that you can use XML tags to make your prompt clearer and more parseable to the model. It turns out, you can also ask the model to **use XML tags to make its output clearer and more easily understandable** to humans.

### Examples

Remember the 'preamble problem' we solved in Chapter 2 by asking the model to skip the preamble entirely? It turns out we can also achieve a similar outcome by **telling the model to put the output in XML tags**.

In [ ]:
# Variable content
ANIMAL = "Rabbit"

# User prompt template with a placeholder for the variable content
USER_PROMPT = f"Please write a haiku about {ANIMAL}. Put the answer in <haiku> XML tags."

# Print the model's response
print("--------------------------- Full prompt with variable substitutions ---------------------------")
print(USER_PROMPT)
print("\n------------------------------------- The model's response -------------------------------------")
print(get_completion(USER_PROMPT))

Why is this something we'd want to do? Having the output in **XML tags lets the
end user reliably extract the haiku and only the haiku by writing a short
program to pull out the content between the tags**.

An extension of this technique is to **put the first opening XML tag in an
artificial `assistant` turn**. Injecting an artificial `assistant` turn tells
the model it has already said something and should continue from there. This is
called "speaking for the model" or "prefilling the model's response."

Below, we've done this with an opening `<haiku>` tag. Notice how the model
continues from where we left off — and, having been started inside the tag, it
will usually close the matching `</haiku>` tag on its own.

In [ ]:
# Variable content
ANIMAL = "Cat"

# User prompt template with a placeholder for the variable content
USER_PROMPT = f"Please write a haiku about: {ANIMAL}."

# Prefill for the model's response
PREFILL = "<haiku>\n"

# Print the model's response
print("--------------------------- Full prompt with variable substitutions ---------------------------")
print("USER TURN:")
print(USER_PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("------------------------------------- The model's response -------------------------------------")
print(get_completion(USER_PROMPT, prefill=PREFILL))

The model can also produce other structured formats — most usefully **JSON**.
A small model won't always get it perfect, but prefilling makes it far more
reliable: by putting an opening brace `{\n` in the assistant turn, you push the
model straight into a JSON object instead of a sentence like "Sure, here's your
JSON…".

In [ ]:
# Variable content
ANIMAL = "Cat"

# User prompt template with a placeholder for the variable content
USER_PROMPT = f"Please write a haiku about: {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Prefill for the model's response
PREFILL = "{\n "

# Print the model's response
print("--------------------------- Full prompt with variable substitutions ---------------------------")
print("USER TURN")
print(USER_PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- The model's response -------------------------------------")
data = get_completion(USER_PROMPT, prefill=PREFILL)
print(data)

Once we have the `data` captured in a variable, we can parse it in software and
use it however we need. Nudging the model into a fixed shape like this is a
lightweight way to get **structured output** — output that follows a
predictable schema, so it's valid, parseable, and ready for downstream code.

One caveat worth knowing: prefilling only *encourages* the format, it doesn't
*guarantee* it — a small model can still emit slightly malformed JSON, so
production code should always wrap `json.loads` in error handling. Some
platforms offer true
[structured-output](https://platform.claude.com/docs/en/build-with-claude/structured-outputs)
features that enforce a schema during generation; that's stronger than this
prefill trick, but the goal is the same.

The model was prompted to reply in JSON, and we used **prefill** — feeding the
model the start of its own answer (here, the opening `{\n ` stored in
`PREFILL`) — to force it straight into the format with no preamble like "Sure!
Here's your JSON". The catch: the model's response continues *from* the prefill,
so `data` holds only the part the model generated, not the `{\n ` we supplied. We
stick them back together before parsing.

In [ ]:
import json

print(f"\n{'='*20} data {'='*20}")
# `data` is only what the model generated — the prefill isn't included
print(data)

# Concatenate the PREFILL we fed in with the output `data`, so we have the complete JSON object
reattached = PREFILL + data

print(f"\n{'='*20} PREFILL + data {'='*20}")
print(reattached)

# Now it's valid JSON, so Python can parse it into a dictionary
parsed = json.loads(reattached)

# Access each field like any Python dict
print(f"\n{'='*40}")
print(f"key: first_line,\nvalue: {parsed['first_line']}")

print(f"\n{'='*40}")
print(f"key: second_line,\nvalue: {parsed['second_line']}")

print(f"\n{'='*40}")
print(f"key: third_line,\nvalue: {parsed['third_line']}")

The output prints three times: first the raw model output (`data`), then the
reassembled object (`PREFILL + data`), then the values pulled out one by one.
That last step is the payoff — once the text is parsed with `json.loads`, it's a
normal Python dictionary, so `parsed['first_line']` gives you the value
directly. That's the whole reason to force structured output: the model's words
become data your program can reliably act on, with no fragile string-hunting.

Below is an example of **multiple input variables in the same prompt AND output formatting specification, all done using XML tags**.

In [ ]:
# First input variable
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Second input variable
ADJECTIVE = "formal"

# User prompt template with a placeholder for the variable content
USER_PROMPT = f"Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Prefill for the model's response (now as an f-string with a variable)
PREFILL = f"<{ADJECTIVE}_email>"

# Print the model's response
print("--------------------------- Full prompt with variable substitutions ---------------------------")
print("USER TURN")
print(USER_PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- The model's response -------------------------------------")
print(get_completion(USER_PROMPT, prefill=PREFILL))

---

## Exercises
- [Exercise 5.1 - Steph Curry GOAT](#exercise-51---steph-curry-goat)
- [Exercise 5.2 - Two Haikus](#exercise-52---two-haikus)
- [Exercise 5.3 - Two Haikus, Two Animals](#exercise-53---two-haikus-two-animals)

### Exercise 5.1 - Steph Curry GOAT
Asked to name the best basketball player of all time, the model usually plays it
safe and names a consensus pick (often Michael Jordan or LeBron James). Can we
steer it somewhere else?

Change **only** the `PREFILL` variable to compel the model to make a detailed
argument that the best basketball player of all time is Stephen Curry. The
prefill is the entire focus of this exercise — leave the other variables alone.

In [ ]:
PREFILL = "" # CHANGE THIS ONLY

USER_PROMPT = "Who is the best basketball player of all time? Please choose one specific player."

# Get the model's response
response = get_completion(USER_PROMPT, prefill=PREFILL)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("Golden State Warriors", text))

# Print the model's response
print("--------------------------- Full prompt with variable substitutions ---------------------------")
print("USER TURN")
print(USER_PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- The model's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", "✅" if grade_exercise(response) else "❌")

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_5_1_hint; print(exercise_5_1_hint)

### Exercise 5.2 - Two Haikus
Modify the `USER_PROMPT` and `PREFILL` variables below using XML tags so that the model writes two haikus about the animal instead of just one. It should be clear where one poem ends and the other begins. Do not modify other variables.

In [ ]:
# Variable content
ANIMAL = "cats"

USER_PROMPT = f"Please write a haiku about {ANIMAL}" # CHANGE THIS
PREFILL = "" # CHANGE THIS

# Get the model's response
response = get_completion(USER_PROMPT, prefill=PREFILL)

# Function to grade exercise correctness
def grade_exercise(text):
    cnt = {}
    for word in text.split():
        cnt[word] = cnt.get(word, 0) + 1
    return bool(
        "cat" in text.lower()
        and cnt.get("<haiku>", 0) >= 1
        and cnt.get("</haiku>", 0) == 2
        and (text.count("\n") + 1) > 5
    )

# Print the model's response
print("--------------------------- Full prompt with variable substitutions ---------------------------")
print("USER TURN")
print(USER_PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- The model's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", "✅" if grade_exercise(response) else "❌")

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_5_2_hint; print(exercise_5_2_hint)

### Exercise 5.3 - Two Haikus, Two Animals
Modify the `USER_PROMPT` and/or `PREFILL` variables below so that **the model produces two haikus about two different animals**. Use `{ANIMAL1}` as a stand-in for the first substitution, and `{ANIMAL2}` as a stand-in for the second substitution. You may only modify the `USER_PROMPT` and `PREFILL` variables.

In [ ]:
# First input variable
ANIMAL1 = "Cat"

# Second input variable
ANIMAL2 = "Dog"

PREFILL = "" # CHANGE THIS
USER_PROMPT = f"Please write a haiku about {ANIMAL1}. Put it in <haiku> tags." # CHANGE THIS

# Get the model's response
response = get_completion(USER_PROMPT, prefill=PREFILL)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("tail", text.lower()) and re.search("cat", text.lower()) and re.search("<haiku>", text))

# Print the model's response
print("--------------------------- Full prompt with variable substitutions ---------------------------")
print("USER TURN")
print(USER_PROMPT)
print(f"\nASSISTANT TURN\n{PREFILL}\n" if PREFILL else "", end="")
print("\n------------------------------------- The model's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", "✅" if grade_exercise(response) else "❌")

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_5_3_hint; print(exercise_5_3_hint)

### Congrats!

If you've solved all exercises up until this point, you're ready to move to the next chapter. Happy prompting!

---

## Example Playground

This is an area for you to experiment freely with the prompt examples shown in this lesson and tweak prompts to see how it may affect the model's responses.

In [ ]:
ANIMAL = "Rabbit"
USER_PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

print("--------------------------- Full prompt with variable substitutions ---------------------------")
print(USER_PROMPT)
print("\n------------------------------------- The model's response -------------------------------------")
print(get_completion(USER_PROMPT))

In [ ]:
ANIMAL = "Cat"
USER_PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

PREFILL = "<haiku>\n"

print("--------------------------- Full prompt with variable substitutions ---------------------------")
print("USER TURN:")
print(USER_PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- The model's response -------------------------------------")
print(get_completion(USER_PROMPT, prefill=PREFILL))

In [ ]:
ANIMAL = "Cat"
USER_PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."
PREFILL = "{\n "

# Print the model's response
print("--------------------------- Full prompt with variable substitutions ---------------------------")
print("USER TURN")
print(USER_PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- The model's response -------------------------------------")
print(get_completion(USER_PROMPT, prefill=PREFILL))

In [ ]:
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."
ADJECTIVE = "formal"
USER_PROMPT = f"Hey the model. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."
PREFILL = f"<{ADJECTIVE}_email>"

print("--------------------------- Full prompt with variable substitutions ---------------------------")
print("USER TURN")
print(USER_PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- The model's response -------------------------------------")
print(get_completion(USER_PROMPT, prefill=PREFILL))